In [0]:
import pandas as pd
import glob
import os

# Folder path where all Excel files are stored
folder_path = r"/Volumes/workspace/default/e-com/cleaned dataset"

# Get all Excel files
files = glob.glob(os.path.join(folder_path, "*.csv"))

# Dictionary to store all DataFrames
dataframes = {}

# Read each file and create separate DataFrame
for file in files:
    
    # Extract file name without extension
    file_name = os.path.splitext(os.path.basename(file))[0]
    
    # Read Excel file
    df = pd.read_csv(file)
    
    # Store in dictionary
    dataframes[file_name] = df

# Access individual DataFrames
category_df   = dataframes["categories"]
coupons_df    = dataframes["coupons"]
cust_df       = dataframes["customers"]
order_items_df= dataframes["order_items"]
orders_df     = dataframes["orders"]
payments_df   = dataframes["payments"]
products_df   = dataframes["products"]
returns_df    = dataframes["returns"]
reviews_df    = dataframes["reviews"]
sellers_df    = dataframes["sellers"]
shipments_df  = dataframes["shipments"]
warehouses_df = dataframes["warehouses"]

category_df


In [0]:
import numpy as np

**Basic Data Audit**

In [0]:
tables = {
    "customers": cust_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "products": products_df,
    "categories": category_df,
    "sellers": sellers_df,
    "warehouses": warehouses_df,
    "payments": payments_df,
    "shipments": shipments_df,
    "returns": returns_df,
    "reviews": reviews_df,
    "coupons": coupons_df
}

for name, df in tables.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum()[df.isna().sum() > 0])

**Cleaning Helper Functions**

In [0]:
def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

def clean_category_text(series):
    return clean_text(series).str.title()

def clean_status(series, default="Unknown"):
    return clean_text(series).str.title().fillna(default)

def parse_date(series):
    return pd.to_datetime(series, errors="coerce")

def add_surrogate_key(df, key_name):
    df = df.reset_index(drop=True).copy()
    df.insert(0, key_name, range(1, len(df) + 1))
    return df

**Clean Source Tables**

In [0]:
cust_df = cust_df.drop_duplicates(subset=["customer_id"]).copy()
orders_df = orders_df.drop_duplicates(subset=["order_id"]).copy()

cust_df["customer_name"] = clean_text(cust_df["customer_name"]).fillna("Unknown Customer")
cust_df["email"] = clean_text(cust_df["email"]).str.lower()
cust_df["city"] = clean_category_text(cust_df["city"])
cust_df["state"] = clean_text(cust_df["state"]).str.upper()
cust_df["customer_segment"] = clean_status(cust_df["customer_segment"])
cust_df["signup_date"] = parse_date(cust_df["signup_date"])

orders_df["order_date"] = parse_date(orders_df["order_date"])
orders_df["order_status"] = clean_status(orders_df["order_status"])
orders_df["sales_channel"] = clean_status(orders_df["sales_channel"])
orders_df["payment_type"] = clean_status(orders_df["payment_type"])
orders_df["shipping_method"] = clean_status(orders_df["shipping_method"])

products_df["product_name"] = clean_text(products_df["product_name"]).fillna("Unknown Product")
products_df["currency"] = clean_text(products_df["currency"]).fillna("INR")
products_df["product_status"] = clean_status(products_df["product_status"])
products_df["created_date"] = parse_date(products_df["created_date"])

category_df["category_name"] = clean_category_text(category_df["category_name"]).fillna("Unknown Category")
category_df["status"] = clean_status(category_df["status"])

sellers_df["seller_name"] = clean_text(sellers_df["seller_name"]).fillna("Unknown Seller")
sellers_df["city"] = clean_category_text(sellers_df["city"])
sellers_df["state"] = clean_text(sellers_df["state"]).str.upper()
sellers_df["seller_status"] = clean_status(sellers_df["seller_status"])
sellers_df["onboarded_date"] = parse_date(sellers_df["onboarded_date"])

warehouses_df["warehouse_name"] = clean_text(warehouses_df["warehouse_name"]).fillna("Unknown Warehouse")
warehouses_df["city"] = clean_category_text(warehouses_df["city"])
warehouses_df["state"] = clean_text(warehouses_df["state"]).str.upper()
warehouses_df["status"] = clean_status(warehouses_df["status"])

payments_df["payment_date"] = parse_date(payments_df["payment_date"])
payments_df["payment_method"] = clean_status(payments_df["payment_method"])
payments_df["payment_status"] = clean_status(payments_df["payment_status"])

shipments_df["shipped_date"] = parse_date(shipments_df["shipped_date"])
shipments_df["delivered_date"] = parse_date(shipments_df["delivered_date"])
shipments_df["carrier"] = clean_status(shipments_df["carrier"])
shipments_df["shipment_status"] = clean_status(shipments_df["shipment_status"])

returns_df["return_date"] = parse_date(returns_df["return_date"])
returns_df["return_reason"] = clean_status(returns_df["return_reason"])
returns_df["return_status"] = clean_status(returns_df["return_status"])

reviews_df["review_date"] = parse_date(reviews_df["review_date"])
reviews_df["rating"] = pd.to_numeric(reviews_df["rating"], errors="coerce")
reviews_df["review_text"] = clean_text(reviews_df["review_text"]).fillna("No Review")

coupons_df["coupon_code"] = clean_text(coupons_df["coupon_code"])
coupons_df["discount_type"] = clean_status(coupons_df["discount_type"])
coupons_df["status"] = clean_status(coupons_df["status"])
coupons_df["start_date"] = parse_date(coupons_df["start_date"])
coupons_df["end_date"] = parse_date(coupons_df["end_date"])

**Create Date Dimension**

In [0]:
all_dates = pd.concat([
    orders_df["order_date"],
    payments_df["payment_date"],
    shipments_df["shipped_date"],
    shipments_df["delivered_date"],
    returns_df["return_date"],
    reviews_df["review_date"],
    cust_df["signup_date"],
    products_df["created_date"],
    sellers_df["onboarded_date"],
    coupons_df["start_date"],
    coupons_df["end_date"]
]).dropna().drop_duplicates().sort_values()

dim_date = pd.DataFrame({"date": all_dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["day"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.day_name()
dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)

dim_date = dim_date[[
    "date_key", "date", "year", "quarter", "month",
    "month_name", "day", "day_name", "week"
]]

display(dim_date)

**Star Schema Dimensions**

In [0]:
dim_customer = cust_df[[
    "customer_id", "customer_name", "email", "phone",
    "city", "state", "customer_segment", "signup_date"
]].copy()
dim_customer = add_surrogate_key(dim_customer, "customer_key")

dim_category = category_df[[
    "category_id", "category_name", "parent_category_id", "status"
]].copy()
dim_category = add_surrogate_key(dim_category, "category_key")

dim_seller = sellers_df[[
    "seller_id", "seller_name", "city", "state",
    "seller_rating", "onboarded_date", "seller_status"
]].copy()
dim_seller = add_surrogate_key(dim_seller, "seller_key")

dim_warehouse = warehouses_df[[
    "warehouse_id", "warehouse_name", "city", "state",
    "capacity_units", "status"
]].copy()
dim_warehouse = add_surrogate_key(dim_warehouse, "warehouse_key")

dim_product = products_df.merge(
    dim_category[["category_key", "category_id", "category_name"]],
    on="category_id",
    how="left"
)

dim_product = dim_product[[
    "product_id", "product_name", "category_key", "category_id",
    "category_name", "seller_id", "list_price", "currency",
    "product_status", "created_date"
]]
dim_product = add_surrogate_key(dim_product, "product_key")

dim_coupon = coupons_df.copy()
dim_coupon = add_surrogate_key(dim_coupon, "coupon_key")

In [0]:
print(dim_category)

**Create Aggregated Supporting Facts**

In [0]:
payment_summary = payments_df.groupby("order_id", as_index=False).agg(
    total_payment_amount=("amount", "sum"),
    payment_count=("payment_id", "count"),
    latest_payment_date=("payment_date", "max"),
    payment_status=("payment_status", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
    payment_method=("payment_method", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
)

shipment_summary = shipments_df.groupby("order_id", as_index=False).agg(
    shipment_count=("shipment_id", "count"),
    total_shipping_cost=("shipping_cost", "sum"),
    first_shipped_date=("shipped_date", "min"),
    latest_delivered_date=("delivered_date", "max"),
    carrier=("carrier", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
    shipment_status=("shipment_status", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
)

return_summary = returns_df.groupby(["order_id", "order_item_id"], as_index=False).agg(
    return_count=("return_id", "count"),
    total_refund_amount=("refund_amount", "sum"),
    latest_return_date=("return_date", "max"),
    return_status=("return_status", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
    return_reason=("return_reason", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
)

review_summary = reviews_df.groupby(["order_id", "product_id", "customer_id"], as_index=False).agg(
    review_count=("review_id", "count"),
    average_rating=("rating", "mean"),
    latest_review_date=("review_date", "max")
)

**Create Sales Fact Table**

In [0]:
fact_sales = order_items_df.merge(orders_df, on="order_id", how="left")

fact_sales = fact_sales.merge(
    dim_customer[["customer_key", "customer_id"]],
    on="customer_id",
    how="left"
)

fact_sales = fact_sales.merge(
    dim_product[["product_key", "product_id"]],
    on="product_id",
    how="left"
)

fact_sales = fact_sales.merge(
    dim_seller[["seller_key", "seller_id"]],
    on="seller_id",
    how="left"
)

fact_sales = fact_sales.merge(
    dim_warehouse[["warehouse_key", "warehouse_id"]],
    on="warehouse_id",
    how="left"
)

fact_sales = fact_sales.merge(
    payment_summary,
    on="order_id",
    how="left"
)

fact_sales = fact_sales.merge(
    shipment_summary,
    on="order_id",
    how="left"
)

fact_sales = fact_sales.merge(
    return_summary,
    on=["order_id", "order_item_id"],
    how="left"
)

fact_sales = fact_sales.merge(
    review_summary,
    on=["order_id", "product_id", "customer_id"],
    how="left"
)

fact_sales["discount_percent"] = fact_sales["discount_percent"].fillna(0)
fact_sales["gross_amount"] = fact_sales["quantity"] * fact_sales["unit_price"]
fact_sales["discount_amount"] = fact_sales["gross_amount"] * fact_sales["discount_percent"] / 100
fact_sales["taxable_amount"] = fact_sales["gross_amount"] - fact_sales["discount_amount"]
fact_sales["tax_amount"] = fact_sales["taxable_amount"] * fact_sales["tax_percent"] / 100
fact_sales["net_sales_amount"] = fact_sales["taxable_amount"] + fact_sales["tax_amount"]

fact_sales["return_count"] = fact_sales["return_count"].fillna(0).astype(int)
fact_sales["total_refund_amount"] = fact_sales["total_refund_amount"].fillna(0)
fact_sales["is_returned"] = np.where(fact_sales["return_count"] > 0, 1, 0)

fact_sales["order_date_key"] = fact_sales["order_date"].dt.strftime("%Y%m%d")
fact_sales["payment_date_key"] = fact_sales["latest_payment_date"].dt.strftime("%Y%m%d")
fact_sales["shipped_date_key"] = fact_sales["first_shipped_date"].dt.strftime("%Y%m%d")
fact_sales["delivered_date_key"] = fact_sales["latest_delivered_date"].dt.strftime("%Y%m%d")
fact_sales["return_date_key"] = fact_sales["latest_return_date"].dt.strftime("%Y%m%d")
fact_sales["review_date_key"] = fact_sales["latest_review_date"].dt.strftime("%Y%m%d")

date_key_cols = [
    "order_date_key", "payment_date_key", "shipped_date_key",
    "delivered_date_key", "return_date_key", "review_date_key"
]

for col in date_key_cols:
    fact_sales[col] = pd.to_numeric(fact_sales[col], errors="coerce").astype("Int64")

fact_sales = fact_sales[[
    "order_item_id", "order_id",
    "customer_key", "product_key", "seller_key", "warehouse_key",
    "order_date_key", "payment_date_key", "shipped_date_key",
    "delivered_date_key", "return_date_key", "review_date_key",
    "quantity", "unit_price", "discount_percent", "tax_percent",
    "gross_amount", "discount_amount", "taxable_amount",
    "tax_amount", "net_sales_amount",
    "total_payment_amount", "payment_count",
    "shipment_count", "total_shipping_cost",
    "return_count", "total_refund_amount", "is_returned",
    "review_count", "average_rating",
    "order_status", "sales_channel", "payment_type",
    "payment_status", "payment_method",
    "shipping_method", "carrier", "shipment_status",
    "return_status", "return_reason"
]]

fact_sales.head()

**Star Schema Tables**

In [0]:
star_schema = {
    "fact_sales": fact_sales,
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_category": dim_category,
    "dim_seller": dim_seller,
    "dim_warehouse": dim_warehouse,
    "dim_coupon": dim_coupon
}

for name, df in star_schema.items():
    print(name, df.shape)

**Snowflake Schema Dimensions**

In [0]:
dim_location = pd.concat([
    cust_df[["city", "state"]],
    sellers_df[["city", "state"]],
    warehouses_df[["city", "state"]]
]).drop_duplicates().reset_index(drop=True)

dim_location = add_surrogate_key(dim_location, "location_key")

dim_customer_snowflake = cust_df.merge(
    dim_location,
    on=["city", "state"],
    how="left"
)[[
    "customer_id", "customer_name", "email", "phone",
    "customer_segment", "signup_date", "location_key"
]]
dim_customer_snowflake = add_surrogate_key(dim_customer_snowflake, "customer_key")

dim_seller_snowflake = sellers_df.merge(
    dim_location,
    on=["city", "state"],
    how="left"
)[[
    "seller_id", "seller_name", "seller_rating",
    "onboarded_date", "seller_status", "location_key"
]]
dim_seller_snowflake = add_surrogate_key(dim_seller_snowflake, "seller_key")

dim_warehouse_snowflake = warehouses_df.merge(
    dim_location,
    on=["city", "state"],
    how="left"
)[[
    "warehouse_id", "warehouse_name", "capacity_units",
    "status", "location_key"
]]
dim_warehouse_snowflake = add_surrogate_key(dim_warehouse_snowflake, "warehouse_key")

dim_parent_category = category_df[[
    "category_id", "category_name"
]].rename(columns={
    "category_id": "parent_category_id",
    "category_name": "parent_category_name"
})

dim_category_snowflake = category_df.merge(
    dim_parent_category,
    on="parent_category_id",
    how="left"
)

dim_category_snowflake = dim_category_snowflake[[
    "category_id", "category_name", "parent_category_id",
    "parent_category_name", "status"
]]
dim_category_snowflake = add_surrogate_key(dim_category_snowflake, "category_key")

dim_product_snowflake = products_df.merge(
    dim_category_snowflake[["category_key", "category_id"]],
    on="category_id",
    how="left"
)

dim_product_snowflake = dim_product_snowflake[[
    "product_id", "product_name", "category_key", "seller_id",
    "list_price", "currency", "product_status", "created_date"
]]
dim_product_snowflake = add_surrogate_key(dim_product_snowflake, "product_key")

**Export Star Schema and Snowflake Schema**

In [0]:
output_path = f"{folder_path}/etl_output"
star_path = f"{output_path}/star_schema"
snowflake_path = f"{output_path}/snowflake_schema"

import os

os.makedirs(star_path, exist_ok=True)
os.makedirs(snowflake_path, exist_ok=True)

for name, df in star_schema.items():
    df.to_csv(f"{star_path}/{name}.csv", index=False)

snowflake_schema = {
    "fact_sales": fact_sales,
    "dim_date": dim_date,
    "dim_customer": dim_customer_snowflake,
    "dim_product": dim_product_snowflake,
    "dim_category": dim_category_snowflake,
    "dim_seller": dim_seller_snowflake,
    "dim_warehouse": dim_warehouse_snowflake,
    "dim_location": dim_location,
    "dim_coupon": dim_coupon
}

for name, df in snowflake_schema.items():
    df.to_csv(f"{snowflake_path}/{name}.csv", index=False)

print("ETL completed successfully.")
print("Star schema saved to:", star_path)
print("Snowflake schema saved to:", snowflake_path)

**Final Validation**

In [0]:
print("Fact Sales Rows:", len(fact_sales))
print("Order Items Rows:", len(order_items_df))

print("\nMissing surrogate keys in fact table:")
display(fact_sales[[
    "customer_key", "product_key", "seller_key", "warehouse_key"
]].isna().sum().to_frame("missing_count"))

print("\nStar Schema:")
for name, df in star_schema.items():
    print(name, df.shape)

print("\nSnowflake Schema:")
for name, df in snowflake_schema.items():
    print(name, df.shape)

In [0]:
# ============================================
# WRITE STAR SCHEMA TO UNITY CATALOG TABLES
# ============================================

import pandas as pd

# Define catalog and schema for Unity Catalog
catalog_name = "workspace"
schema_name = "default"

# Convert pandas DataFrames to Spark DataFrames and write to Unity Catalog
tables_to_write = {
    'dim_date': dim_date,
    'dim_customer': dim_customer,
    'dim_category': dim_category,
    'dim_seller': dim_seller,
    'dim_warehouse': dim_warehouse,
    'dim_product': dim_product,
    'dim_coupon': dim_coupon,
    'fact_sales': fact_sales
}

for table_name, df in tables_to_write.items():
    try:
        # Convert pandas DataFrame to Spark DataFrame
        spark_df = spark.createDataFrame(df)
        
        # Write to Unity Catalog
        full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
        spark_df.write.mode("overwrite").saveAsTable(full_table_name)
        
        print(f"✓ {full_table_name} created successfully ({df.shape[0]} rows, {df.shape[1]} columns)")
    except Exception as e:
        print(f"✗ Error creating {table_name}: {e}")

print("\n" + "="*60)
print("Star Schema tables written to Unity Catalog successfully")
print(f"Location: {catalog_name}.{schema_name}")
print("="*60)

-------

fact_sales (65000, 40)
dim_date (2178, 9)
dim_customer (15000, 9)
dim_product (5000, 11)
dim_category (84, 5)
dim_seller (800, 8)
dim_warehouse (60, 7)
dim_coupon (500, 8)

**checking**

In [0]:
%pip install sqlalchemy oracledb

In [0]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
from sqlalchemy import create_engine, text

# ============================================================
# ORACLE DATABASE CONNECTION DETAILS
# ============================================================

username = "your_oracle_username"  # TODO: replace with your real Oracle username before running (or use dbutils.secrets.get(...))
password = "your_oracle_password"  # TODO: replace with your real Oracle password before running
host = "Hostname"
port = 1521
service_name = "orcl"

# ============================================================
# CREATE ORACLE CONNECTION (THIN MODE - NO CLIENT REQUIRED)
# ============================================================

connection_string = f"oracle+oracledb://{username}:{password}@{host}:{port}/?service_name={service_name}"
engine = create_engine(connection_string)

print("Oracle engine created successfully")

# ============================================================
# TEST CONNECTION
# ============================================================

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 'Connected!' as status FROM DUAL"))
        print("✓ Oracle connection successful:", result.fetchone()[0])
except Exception as e:
    print("✗ Connection failed:", e)
    raise

# ============================================================
# DROP EXISTING TABLES
# ============================================================

with engine.connect() as conn:
    tables = [
        "fact_sales", "dim_date", "dim_customer", "dim_product",
        "dim_category", "dim_seller", "dim_warehouse", "dim_coupon"
    ]
    for table in tables:
        try:
            conn.execute(text(f"DROP TABLE {table} CASCADE CONSTRAINTS"))
            print(f"✓ Dropped {table}")
        except Exception:
            print(f"  {table} does not exist (skipping)")
    conn.commit()

print("\n" + "="*60)
print("Creating tables and inserting data...")
print("="*60 + "\n")

# ============================================================
# CREATE AND INSERT DIM_DATE
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_date (
            date_key NUMBER PRIMARY KEY,
            date_value DATE,
            year NUMBER,
            quarter NUMBER,
            month NUMBER,
            month_name VARCHAR2(20),
            day NUMBER,
            day_name VARCHAR2(20),
            week NUMBER
        )
    """))
    conn.commit()
    print("✓ Created dim_date")

dim_date_export = dim_date.rename(columns={'date': 'date_value'})
dim_date_export.to_sql('dim_date', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_date)} rows into dim_date")

# ============================================================
# CREATE AND INSERT DIM_CUSTOMER
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_customer (
            customer_key NUMBER PRIMARY KEY,
            customer_id VARCHAR2(50),
            customer_name VARCHAR2(200),
            email VARCHAR2(200),
            phone VARCHAR2(50),
            city VARCHAR2(100),
            state VARCHAR2(50),
            customer_segment VARCHAR2(100),
            signup_date DATE
        )
    """))
    conn.commit()
    print("✓ Created dim_customer")

dim_customer.to_sql('dim_customer', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_customer)} rows into dim_customer")

# ============================================================
# CREATE AND INSERT DIM_PRODUCT
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_product (
            product_key NUMBER PRIMARY KEY,
            product_id VARCHAR2(50),
            product_name VARCHAR2(300),
            category_key NUMBER,
            category_id VARCHAR2(50),
            category_name VARCHAR2(200),
            seller_id VARCHAR2(50),
            list_price NUMBER,
            currency VARCHAR2(10),
            product_status VARCHAR2(50),
            created_date DATE
        )
    """))
    conn.commit()
    print("✓ Created dim_product")

dim_product.to_sql('dim_product', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_product)} rows into dim_product")

# ============================================================
# CREATE AND INSERT DIM_CATEGORY
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_category (
            category_key NUMBER PRIMARY KEY,
            category_id VARCHAR2(50),
            category_name VARCHAR2(200),
            parent_category_id VARCHAR2(50),
            status VARCHAR2(50)
        )
    """))
    conn.commit()
    print("✓ Created dim_category")

dim_category.to_sql('dim_category', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_category)} rows into dim_category")

# ============================================================
# CREATE AND INSERT DIM_SELLER
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_seller (
            seller_key NUMBER PRIMARY KEY,
            seller_id VARCHAR2(50),
            seller_name VARCHAR2(200),
            city VARCHAR2(100),
            state VARCHAR2(50),
            seller_rating NUMBER,
            onboarded_date DATE,
            seller_status VARCHAR2(50)
        )
    """))
    conn.commit()
    print("✓ Created dim_seller")

dim_seller.to_sql('dim_seller', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_seller)} rows into dim_seller")

# ============================================================
# CREATE AND INSERT DIM_WAREHOUSE
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_warehouse (
            warehouse_key NUMBER PRIMARY KEY,
            warehouse_id VARCHAR2(50),
            warehouse_name VARCHAR2(200),
            city VARCHAR2(100),
            state VARCHAR2(50),
            capacity_units NUMBER,
            status VARCHAR2(50)
        )
    """))
    conn.commit()
    print("✓ Created dim_warehouse")

dim_warehouse.to_sql('dim_warehouse', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_warehouse)} rows into dim_warehouse")

# ============================================================
# CREATE AND INSERT DIM_COUPON
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE dim_coupon (
            coupon_key NUMBER PRIMARY KEY,
            coupon_id VARCHAR2(50),
            coupon_code VARCHAR2(50),
            discount_type VARCHAR2(50),
            discount_value NUMBER,
            start_date DATE,
            end_date DATE,
            status VARCHAR2(50)
        )
    """))
    conn.commit()
    print("✓ Created dim_coupon")

dim_coupon.to_sql('dim_coupon', con=engine, if_exists='append', index=False, chunksize=500)
print(f"  Inserted {len(dim_coupon)} rows into dim_coupon")

# ============================================================
# CREATE AND INSERT FACT_SALES
# ============================================================

with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE fact_sales (
            order_item_id VARCHAR2(50) PRIMARY KEY,
            order_id VARCHAR2(50),
            customer_key NUMBER,
            product_key NUMBER,
            seller_key NUMBER,
            warehouse_key NUMBER,
            order_date_key NUMBER,
            payment_date_key NUMBER,
            shipped_date_key NUMBER,
            delivered_date_key NUMBER,
            return_date_key NUMBER,
            review_date_key NUMBER,
            quantity NUMBER,
            unit_price NUMBER,
            discount_percent NUMBER,
            tax_percent NUMBER,
            gross_amount NUMBER,
            discount_amount NUMBER,
            taxable_amount NUMBER,
            tax_amount NUMBER,
            net_sales_amount NUMBER,
            total_payment_amount NUMBER,
            payment_count NUMBER,
            shipment_count NUMBER,
            total_shipping_cost NUMBER,
            return_count NUMBER,
            total_refund_amount NUMBER,
            is_returned NUMBER,
            review_count NUMBER,
            average_rating NUMBER,
            order_status VARCHAR2(50),
            sales_channel VARCHAR2(50),
            payment_type VARCHAR2(50),
            payment_status VARCHAR2(50),
            payment_method VARCHAR2(50),
            shipping_method VARCHAR2(50),
            carrier VARCHAR2(50),
            shipment_status VARCHAR2(50),
            return_status VARCHAR2(50),
            return_reason VARCHAR2(200)
        )
    """))
    conn.commit()
    print("✓ Created fact_sales")

fact_sales.to_sql('fact_sales', con=engine, if_exists='append', index=False, chunksize=1000)
print(f"  Inserted {len(fact_sales)} rows into fact_sales")

print("\n" + "="*60)
print("✓ All star schema tables created and data inserted successfully!")
print(f"✓ Connected to Oracle: {host}:{port}/{service_name}")
print("="*60)

engine.dispose()
print("\n✓ Oracle connection closed")

In [0]:
for name, df in star_schema.items():
    print(f"{name}: {len(df)}")

For your ETL/Data Warehouse project, the most important checks are:

✅ Tables exist
✅ Row counts match DataFrames
✅ Foreign keys exist
✅ Join query returns data
✅ No orphan records

An orphan record is a child-table record whose foreign key does not have a matching primary key in the parent table. Orphan records cause data integrity issues and can lead to missing data in joins and reports.